# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ravikiranbathe/flyrank-ai/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — The Anatomy of Growing Content

The paper compares pages with rising impressions against pages with falling impressions. It found that growing pages were generally longer and younger: growing pages averaged about 3,180 words and 184 days old, compared with about 2,311 words and 230 days for declining pages. The paper describes this as an observational comparison, not proof that longer or newer content directly causes growth.

**My methodology question:** How is the “growing” or “declining” label defined, and how are the time windows separated from the other measurements? The paper defines the trend using the change in impressions over 30 days compared with the previous 30 days. I would want to confirm that the word count and age measurements are available independently of that trend calculation and that short-term changes or seasonality are not driving the classification.

I think this would help clarify how strongly the observed difference between growing and declining pages can be interpreted.
### Finding 2 — The Freshness Multiplier

The paper found that 365+ day-old content that had been refreshed within 30 days showed a much stronger performance in the dataset. The comparison reported a 3.2× health increase, from 10.7 to 34.5, and 57× more impressions, from 71 to 4,039. The paper presents this as one of the strongest measured levers in the portfolio.

**My methodology question:** I would like to know how the pages selected for refreshing differed from pages that were not refreshed. For example, were refreshed pages already getting more traffic, or were they selected because they had better potential? If so, some of the observed difference could be related to which pages were chosen for refreshing rather than the refresh itself. A comparison with similar pages that were not refreshed, along with performance measured before and after the refresh, would make the result easier to interpret.

I would treat this as a question about the strength of the evidence, rather than saying that the paper's result is wrong.

### My overall takeaway

The paper does a good job of separating observed patterns from stronger claims. Its main findings are based on direct portfolio comparisons, while the ML results are presented as exploratory.

For my own model, I want to follow the same approach: clearly separate what I measured from what I think the result might mean, check for leakage, and avoid claiming more than my validation supports.


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Honest Split

In Week 5, I used a random row-level train/test split. The Random Forest gave an F1 score of 0.9998 with this split.

For this audit, I also used a client-grouped split. The dataset contains 32 clients, so I kept clients separate between the training and test sets. This gives a more realistic check of how the model behaves on clients that were not used for training.

I compare the two results below. The purpose is not to find a better number, but to check whether the validation result changes when the split is made more realistic.

In [10]:
import os
import pandas as pd

# Check current location
print("Current folder:", os.getcwd())

# Clone the repo if it is not already available
if not os.path.exists("flyrank-ai"):
    !git clone https://github.com/Ravikiranbathe/flyrank-ai.git

# Move into the repository
os.chdir("flyrank-ai")

print("Repository folder:", os.getcwd())

# Load dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)

Current folder: /content/flyrank-ai
Cloning into 'flyrank-ai'...
remote: Enumerating objects: 154, done.
remote: Counting objects: 100% (154/154), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 154 (delta 61), reused 101 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (154/154), 1.86 MiB | 7.82 MiB/s, done.
Resolving deltas: 100% (61/61), done.
Repository folder: /content/flyrank-ai/flyrank-ai
Dataset shape: (30000, 44)


In [11]:
# Create the same target used in Week 5

df["baseline_score"] = 0

df.loc[df["content_age_days"] >= 365, "baseline_score"] += 40
df.loc[df["trend_pct"] < -10, "baseline_score"] += 25
df.loc[df["ctr"] < 2, "baseline_score"] += 20
df.loc[df["impressions_90d"] >= 1000, "baseline_score"] += 15

df["target"] = (df["baseline_score"] >= 60).astype(int)

# Week 5 features
features = [
    "content_age_days",
    "trend_pct",
    "ctr",
    "impressions_90d"
]

X = df[features].fillna(0)
y = df["target"]
groups = df["client_id"]

print("Features:", features)
print("Target positive rate:", round(y.mean(), 4))
print("Number of clients:", groups.nunique())

Features: ['content_age_days', 'trend_pct', 'ctr', 'impressions_90d']
Target positive rate: 0.4641
Number of clients: 32


In [12]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# -----------------------------
# 1. Original Week-5 split
# -----------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

rf_random = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_random.fit(X_train, y_train)

y_pred_random = rf_random.predict(X_test)

random_results = {
    "Split": "Random row split",
    "Accuracy": accuracy_score(y_test, y_pred_random),
    "Precision": precision_score(y_test, y_pred_random),
    "Recall": recall_score(y_test, y_pred_random),
    "F1": f1_score(y_test, y_pred_random)
}

# -----------------------------
# 2. Client-grouped split
# -----------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

rf_grouped = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_grouped.fit(X_train_group, y_train_group)

y_pred_grouped = rf_grouped.predict(X_test_group)

grouped_results = {
    "Split": "Client-grouped split",
    "Accuracy": accuracy_score(y_test_group, y_pred_grouped),
    "Precision": precision_score(y_test_group, y_pred_grouped),
    "Recall": recall_score(y_test_group, y_pred_grouped),
    "F1": f1_score(y_test_group, y_pred_grouped)
}

# -----------------------------
# 3. Before vs. after table
# -----------------------------

validation_comparison = pd.DataFrame([
    random_results,
    grouped_results
]).round(4)

print("Validation comparison:")
display(validation_comparison)

print("Overall target positive rate:", round(y.mean(), 4))
print("Random test positive rate:", round(y_test.mean(), 4))
print("Grouped test positive rate:", round(y_test_group.mean(), 4))
print("Grouped test clients:", groups.iloc[test_idx].nunique())

Validation comparison:


,Split,Accuracy,Precision,Recall,F1
0,Random row split,0.9998,0.9996,1.0,0.9998
1,Client-grouped split,1.0000,1.0000,1.0,1.0000


Overall target positive rate: 0.4641
Random test positive rate: 0.4562
Grouped test positive rate: 0.4735
Grouped test clients: 7


### What changed?

The random row split gave an F1 score of 0.9998, while the client-grouped split gave an F1 score of 1.0000.

In this experiment, the grouped split did not reduce the measured performance. However, I would not interpret this as proof that the model generalizes perfectly to new clients.

The main reason is that my target was created using the same four features given to the Random Forest. Because of this, the model is mainly learning to reproduce the baseline rule used to create the target.

My main takeaway is that the honest split did not show a performance drop, but the validation result still needs to be interpreted carefully. The connection between the target and features is the main issue I need to document in the leakage audit.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage Audit

I checked the features used by my Week-5 Random Forest against the way the target was created.

The target is created from `baseline_score`, where a page gets a positive label when the score is 60 or higher.

The baseline score uses the same four features used by my model:

- `content_age_days`
- `trend_pct`
- `ctr`
- `impressions_90d`

This means the model features are directly connected to the target construction. I consider this a label-derived feature issue.

Because of this, the very high F1 score should not be treated as evidence that the model independently predicts real-world refresh outcomes. It mainly shows that the Random Forest can reproduce the rule used to create the target.

For a stronger future version, I would use an independently observed outcome as the target and make sure the features are available before that outcome is measured.

In [13]:
# Feature leakage audit

leakage_audit = pd.DataFrame({
    "Feature": features,
    "Used_to_create_target": [True, True, True, True],
    "Potential_issue": [
        "Used in baseline_score",
        "Used in baseline_score",
        "Used in baseline_score",
        "Used in baseline_score"
    ]
})

display(leakage_audit)

,Feature,Used_to_create_target,Potential_issue
0,content_age_days,True,Used in baseline_score
1,trend_pct,True,Used in baseline_score
2,ctr,True,Used in baseline_score
3,impressions_90d,True,Used in baseline_score


In [14]:
# Check for actual model errors in the grouped test set

error_examples = df.iloc[test_idx].copy()

error_examples["actual"] = y_test_group.values
error_examples["predicted"] = y_pred_grouped

errors = error_examples[
    error_examples["actual"] != error_examples["predicted"]
]

print("Number of grouped-split errors:", len(errors))

if len(errors) > 0:
    display(
        errors[
            [
                "client_id",
                "content_age_days",
                "trend_pct",
                "ctr",
                "impressions_90d",
                "actual",
                "predicted"
            ]
        ].head(10)
    )
else:
    print("No classification errors were found in this grouped test split.")

Number of grouped-split errors: 0
No classification errors were found in this grouped test split.


In [15]:
# Representative examples near the target threshold

audit_examples = df.iloc[test_idx].copy()

# Distance from the baseline-score threshold of 60
audit_examples["distance_from_threshold"] = (
    audit_examples["baseline_score"] - 60
).abs()

# Show examples closest to the decision boundary
borderline_examples = audit_examples.sort_values(
    "distance_from_threshold"
).head(10)

display(
    borderline_examples[
        [
            "client_id",
            "content_age_days",
            "trend_pct",
            "ctr",
            "impressions_90d",
            "baseline_score",
            "target"
        ]
    ]
)

,client_id,content_age_days,trend_pct,ctr,impressions_90d,baseline_score,target
13783,client_4e07408562,487,-7.3,0.00,464,60,1
13984,client_434c9b5ae5,105,-58.0,0.14,2811,60,1
13998,client_f369cb89fc,106,-31.5,0.07,1502,60,1
3669,client_8527a891e2,223,-14.8,0.07,8076,60,1
3677,client_e629fa6598,460,50.0,0.00,10,60,1
3690,client_4e07408562,537,18.6,0.00,402,60,1
3732,client_f369cb89fc,180,-10.6,0.40,1003,60,1
13902,client_e629fa6598,502,NaN,0.00,11,60,1
13908,client_4e07408562,326,-90.3,0.19,9238,60,1
13929,client_4e07408562,280,-34.1,0.00,1258,60,1


### Error and failure review

The client-grouped test produced zero classification errors, so there are no genuine model misclassification examples to report from this split. I have not created artificial failure examples.

Instead, I inspected examples close to the baseline-score threshold. The examples shown above all have a baseline score of 60, which is exactly the threshold used to create the positive target.

These examples are useful because they show that the model is working around a rule-defined boundary. A small change in one of the input signals could potentially move a page across that boundary.

This reinforces the main limitation of the experiment: the perfect measured classification result is largely a consequence of the target being constructed from the same features used by the model.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Before

My Week-5 model accurately identifies pages that should be refreshed and can be used to prioritize content refresh opportunities.

### After

In this dataset, the Random Forest reproduced the rule-based refresh score with very high measured performance under both a random row split and a client-grouped split. Because the target was constructed from the same four features used by the model, I treat this result as a directional decision-support experiment rather than evidence that the model independently predicts future refresh outcomes.

The model can therefore be described as a way to rank or flag pages for review based on the measured signals in this dataset. Further validation with an independently observed outcome would be needed before making stronger claims about real-world refresh impact.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.